In [17]:
!pip install -qU langchain langchain-openai

In [18]:
import os
from langchain.chat_models import init_chat_model
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
model = init_chat_model("gpt-4o-mini")

In [19]:
from langchain.tools import tool

#create booking hotel tool
@tool
def book_hotel(hotel_name: str):
    """Book a hotel"""
    return f"Successfully booked a stay at {hotel_name}."

#create flight hotel tool
@tool
def book_flight(from_airport: str, to_airport: str):
    """Book a flight"""
    return f"Successfully booked a flight from {from_airport} to {to_airport}."

In [20]:
from langchain.agents import create_agent

#create hotel agent
flight_assistant = create_agent(
    model=model,
    tools=[book_flight],
    system_prompt="You are a flight booking assistant",
    name="flight_assistant"
)

#create hotel agent
hotel_assistant = create_agent(
    model=model,
    tools=[book_hotel],
    system_prompt="You are a hotel booking assistant",
    name="hotel_assistant"
)

In [21]:
flight_assistant.invoke({"messages": [{"role": "user", "content": "Book a flight from CGK to CDG."}]})

{'messages': [HumanMessage(content='Book a flight from CGK to CDG.', additional_kwargs={}, response_metadata={}, id='6cd3b7ed-5cc7-40c2-833d-4f27ccbc74dd'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 67, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_29330a9688', 'id': 'chatcmpl-CpYjOiJM3lw9dEw51BL47BhDJ017G', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, name='flight_assistant', id='lc_run--019b45e6-27a3-7d51-93d7-d4d03d25550a-0', tool_calls=[{'name': 'book_flight', 'args': {'from_airport': 'CGK', 'to_airport': 'CDG'}, 'id': 'call_OlXa4tM202VAHEiX63qPUx59', 'type': 'tool_call'}], usage_metadat

In [22]:
@tool
def flight_agent(from_airport: str, to_airport: str):
    """Book a flight"""
    return flight_assistant.invoke({"messages": [{"role": "user", "content": f"Book a flight from {from_airport} to {to_airport}."}]})

@tool
def hotel_agent(hotel_name: str):
    """Book a hotel"""
    return hotel_assistant.invoke({"messages": [{"role": "user", "content": f"Book a stay at {hotel_name}."}]})

In [23]:
supervisor = create_agent(
    tools=[flight_agent, hotel_agent], #list of agents that will helps supervisor agent
    model=model, #model for supervisor agent
    system_prompt= "You manage a hotel booking assistant and a flight booking assistant. Assign work to them." #prompt for supervisor agent
)

In [24]:
query = (
    "book a flight from BOS to JFK and a stay at McKittrick Hotel."
)

for step in supervisor.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  flight_agent (call_iU3SY9w0hGPtDoYCUmAZAs3X)
 Call ID: call_iU3SY9w0hGPtDoYCUmAZAs3X
  Args:
    from_airport: BOS
    to_airport: JFK
  hotel_agent (call_eG4CttjkJtpA1TqarRPNLD0o)
 Call ID: call_eG4CttjkJtpA1TqarRPNLD0o
  Args:
    hotel_name: McKittrick Hotel
================================= Tool Message =================================
Name: flight_agent

{'messages': [HumanMessage(content='Book a flight from BOS to JFK.', additional_kwargs={}, response_metadata={}, id='d3e8c322-3d94-40c8-bfc8-b54c2f3ee250'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 65, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': '

In [28]:
response = supervisor.invoke(
    {"messages": [{"role": "user", "content": query}]}
)
answer = response["messages"][-1].content
print(answer)

Your bookings have been successfully completed:

- **Flight:** Your flight from Boston (BOS) to New York (JFK) has been successfully booked. Safe travels!
- **Hotel:** Your stay at the McKittrick Hotel has also been successfully booked. If you need any further assistance, feel free to ask!
